# Phase 3 &mdash; Bronze Layer

This is the first stage of our Medallion pipeline. Its job is to take the Lending Club loan data exactly as it arrived and register it as a managed Delta table. No cleaning, no filtering, no type changes &mdash; that all happens in the Silver layer. Keeping Bronze untouched means we always have a faithful record of the source, and that any bug we introduce later can be traced back by re-reading the raw table.

### Input
The 2014&ndash;2017 Lending Club CSV produced by `src/01_data_loading.py`. We uploaded it through the Databricks Catalog UI, which automatically converted the upload into a managed Delta table at `workspace.default.optimized_data_14_17`.

### Output
`workspace.default.bronze_loans` &mdash; an identical copy of the uploaded table, renamed under our medallion convention.

### Why a sanitization step?
The pandas export that produced the CSV left an `Unnamed: 0` index column. Delta rejects column names containing spaces, colons, or other punctuation, so we run a one-pass rename that replaces any non-alphanumeric character with an underscore and lowercases every name. It is a *cosmetic* change &mdash; the row values are identical to the uploaded data &mdash; but it lets Delta accept the schema.

In [ ]:
# A SparkSession named `spark` is already available on a Databricks cluster.
# The try/except keeps the notebook runnable outside Databricks too (e.g. in a
# local `pip install pyspark delta-spark` environment for development).
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder.appName("phase3-bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

print(f"Spark {spark.version} ready.")

In [ ]:
# Source and destination tables. Update SOURCE_TABLE if the uploaded CSV landed
# somewhere other than workspace.default.
SOURCE_TABLE = "workspace.default.optimized_data_14_17"
BRONZE_TABLE = "workspace.default.bronze_loans"

## Step 1 &mdash; Load the uploaded table

No transformations. Just read it into a Spark DataFrame and confirm the shape.

In [ ]:
raw_df = spark.table(SOURCE_TABLE)
print(f"Rows:    {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns)}")

## Step 2 &mdash; Sanity-check with a few rows and the schema

In [ ]:
raw_df.limit(5).toPandas()

In [ ]:
raw_df.printSchema()

## Step 3 &mdash; Sanitize column names

Replace any character outside `[A-Za-z0-9_]` with `_` and lowercase the result. In our data this only renames `Unnamed: 0` (a pandas index artifact) to `unnamed_0`, but the same rule safely handles anything else.

In [ ]:
import re

def sanitize(name):
    cleaned = re.sub(r"[^A-Za-z0-9_]+", "_", name).strip("_").lower()
    return cleaned or "col"

renamed_df = raw_df
changed = []
for old in raw_df.columns:
    new = sanitize(old)
    if new != old:
        renamed_df = renamed_df.withColumnRenamed(old, new)
        changed.append((old, new))

print(f"Renamed {len(changed)} column(s):")
for old, new in changed:
    print(f"  {old!r:<20} -> {new!r}")

## Step 4 &mdash; Write the Bronze Delta table

`mode("overwrite")` makes re-runs idempotent &mdash; the table ends in the same state whether this is the first run or the tenth.

In [ ]:
(
    renamed_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Wrote Delta table: {BRONZE_TABLE}")

## Step 5 &mdash; Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {BRONZE_TABLE}").show()
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}") \
    .select("name", "location", "numFiles", "sizeInBytes") \
    .show(truncate=False)